# Roads and Bridges Cracks · YOLO26s · Google Colab

Dataset: **Roads and Bridges Cracks (YOLOv8 Format)** yang lengkap, satu kelas **`cracks`**. Semua jenis permukaan dari sumber dipakai, termasuk aspal, beton, dan struktur lain. Tidak ada filter khusus aspal dan tidak ada foto archive 7.

**Format YOLOv8 tidak membatasi versi model.** Notebook ini memakai `yolo26s.pt` untuk deteksi bbox. Sumber mencampur 1.141 anotasi bbox dan 2.811 poligon; seluruh label dalam ZIP siap training telah diseragamkan menjadi bbox YOLO lima kolom dengan ID kelas `0`. Anotasi poligon diubah menjadi bbox min/max, sedangkan gambar dan label asli disimpan tanpa perubahan byte.

Dari **3,458 foto sumber**, terdapat **3,439 foto positif unik** untuk training/evaluasi setelah 18 label kosong yang belum terverifikasi sebagai background dan satu duplikat disisihkan. Semua gambar berlabel yang valid dipakai, tanpa pemilahan material permukaan.

Target split **80% train, 10% validasi, 10% test**, seed 42. Seluruh augmentasi dari satu foto asal dan pasangan terdeteksi mirip tetap satu split. Tabel jumlah aktual ditampilkan sebelum training. Skema A dan B memakai pembagian identik.

1. Aktifkan runtime **GPU** di Colab.
2. Upload **`roads_bridges_cracks_yolo26.zip`** ke `MyDrive/deteksi-jalan/data/`. Pakai ZIP siap training ini agar label bbox-nya sudah seragam.
3. Jalankan sel berurutan. Default **Skema A**; B opsional.
4. Model dan metrik tersimpan di folder Drive khusus `yolo_roads_bridges_cracks`.

Default memakai split dalam ZIP. Untuk acak ulang, set `RESPLIT=True`, ganti `SPLIT_SEED` dan `RUN_SUFFIX`. Sumber dan seed yang identik menghasilkan split identik. Notebook menolak ZIP subset aspal maupun dataset gabungan.

Ini **fine-tuning YOLO26s**. Augmentasi offline sumber tetap satu kelompok, termasuk jika kelompok masuk validasi/test. Pemeriksaan kemiripan mengurangi risiko kebocoran tetapi tidak menjamin independensi seluruh lokasi/adegan. Jumlah kelompok asal tersedia di manifest. Semantik anotasi mengikuti sumber dan belum dianotasi ulang. Nilai 0,50 pada dataset sebelumnya belum cukup untuk memastikan penyebabnya; skor pada dataset ini belum diketahui sebelum training.


In [ ]:
# 0 · Pustaka. PyTorch/CUDA mengikuti runtime Colab yang tersedia.
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

ULTRALYTICS_VERSION = '8.4.202'
try:
    installed_yolo = version('ultralytics')
except PackageNotFoundError:
    installed_yolo = None
packages = []
if installed_yolo != ULTRALYTICS_VERSION:
    packages.append(f'ultralytics=={ULTRALYTICS_VERSION}')
for module, package in [
    ('pandas', 'pandas'), ('matplotlib', 'matplotlib'), ('yaml', 'PyYAML'),
    ('cv2', 'opencv-python'), ('PIL', 'Pillow'), ('tqdm', 'tqdm'),
    ('albumentations', 'albumentations>=2,<3'),
]:
    if importlib.util.find_spec(module) is None:
        packages.append(package)
if packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Pustaka siap. Jika runtime meminta restart, restart lalu jalankan lagi dari awal.')


In [ ]:
# 1 · Import dan verifikasi GPU.
from pathlib import Path, PurePosixPath
import gc
import hashlib
import json
import random
import shutil
import tempfile
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
import torch
import ultralytics
from ultralytics import YOLO
import yaml

if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError('Ultralytics lama sudah diimpor. Restart runtime dan jalankan dari awal.')
if not torch.cuda.is_available():
    raise RuntimeError('Pilih runtime GPU di Colab sebelum menjalankan training.')
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE))
print('PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)


In [ ]:
# 2 · Google Drive, sumber data, split, dan konfigurasi training.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/deteksi-jalan')
DATA_ZIP = DRIVE_PROJECT_DIR / 'data' / 'roads_bridges_cracks_yolo26.zip'
WORK_DIR = Path('/content/deteksi-jalan/roads_bridges_cracks')
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SOURCES = {'archive6'}
EXPECTED_IMAGES = 3439
EXPECTED_DATASET_ID = 'roads_bridges_cracks_full'

RESPLIT = False       # False = pakai split tersimpan. True = bagi ulang semua foto.
SPLIT_SEED = 42       # Ganti seed dan RESPLIT=True untuk pembagian baru.
SPLIT_RATIOS = (0.80, 0.10, 0.10)  # train, valid, test; jumlah harus 1.
TRAIN_SEED = 42       # Seed training terpisah dari seed pembagian data.

MODEL = 'yolo26s.pt'
CLASS_NAMES = ['cracks']
SKEMA_LIST = ['A']    # Ganti menjadi ['A', 'B'] untuk menjalankan dua skema.
RUN_SUFFIX = '01'     # Ganti untuk percobaan baru; hasil lama tidak ditimpa.
IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 45
PATIENCE = 10
WORKERS = 2
CLOSE_MOSAIC = 5
PREDICT_CONF = 0.25
RUNS_ROOT = DRIVE_PROJECT_DIR / 'hasil' / 'yolo_roads_bridges_cracks'

if not DATA_ZIP.is_file():
    raise FileNotFoundError(f'Simpan ZIP dataset di {DATA_ZIP}, atau ubah DATA_ZIP sesuai lokasi file.')
if not SKEMA_LIST or len(set(SKEMA_LIST)) != len(SKEMA_LIST) or not set(SKEMA_LIST) <= {'A', 'B'}:
    raise ValueError("SKEMA_LIST harus ['A'], ['B'], atau ['A', 'B'].")
import re
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_SUFFIX):
    raise ValueError('RUN_SUFFIX hanya boleh berisi huruf, angka, garis bawah, atau tanda hubung.')
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)


## 3. Fungsi validasi dan pengacakan kelompok

Sel berikut menyertakan seluruh fungsi persiapan data agar notebook dapat berjalan tanpa mengunggah skrip Python tambahan. Fungsi menolak label/gambar yang berubah atau tidak valid; jumlah foto tidak dikurangi diam-diam. Mode `RESPLIT=True` menggabungkan kembali semua split sebelum pembagian baru, lalu menyimpan manifest dan laporan verifikasi.


In [ ]:
"""Split ulang dataset crack YOLO berdasarkan kelompok asal dan kemiripan gambar.

Modul ini juga disertakan langsung dalam notebook Colab. Tidak menjalankan model.
"""

from collections import Counter, defaultdict
import csv
import hashlib
from io import BytesIO
import json
import math
from pathlib import Path, PurePosixPath
import random
import re
import shutil
import zipfile

import cv2
import numpy as np
from PIL import Image

SPLIT_NAMES = ("train", "valid", "test")


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


def save_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()


def validate_boxes(raw):
    boxes = []
    for line in raw.decode("utf-8-sig").splitlines():
        if not line.strip():
            continue
        values = tuple(map(float, line.split()))
        if len(values) != 5 or values[0] != 0 or not all(math.isfinite(x) for x in values):
            raise ValueError("Label harus lima kolom bbox, satu kelas: 0 = cracks")
        _, x, y, w, h = values
        if min(w, h) <= 0 or min(x-w/2, y-h/2) < -1e-8 or max(x+w/2, y+h/2) > 1+1e-8:
            raise ValueError("Bounding box tidak valid")
        boxes.append(values)
    if not boxes:
        raise ValueError("Dataset positif ini tidak menerima label kosong")
    return boxes


def image_features(raw):
    with Image.open(BytesIO(raw)) as opened:
        opened.load()
        if opened.getexif().get(274, 1) != 1:
            raise ValueError("Orientasi EXIF perlu ditinjau sebelum pemakaian bbox")
        if min(opened.size) < 32 or getattr(opened, "n_frames", 1) != 1:
            raise ValueError("Resolusi terlalu kecil atau gambar multiframe")
        rgb = opened.convert("RGB")
    width, height = rgb.size
    pixel_hash = hashlib.sha256(f"{width}x{height}:".encode() + rgb.tobytes()).hexdigest()
    gray = rgb.convert("L").resize((64, 64), Image.Resampling.LANCZOS)
    thumb = np.asarray(gray, dtype=np.float32)
    small = np.asarray(gray.resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
    phashes = []
    for base in (small, np.fliplr(small)):
        for turns in range(4):
            coefficients = cv2.dct(np.ascontiguousarray(np.rot90(base, turns)))[:8, :8].ravel()
            bits = coefficients > np.median(coefficients[1:])
            bits[0] = False
            phashes.append(sum(int(bit) << index for index, bit in enumerate(bits)))
    return width, height, pixel_hash, tuple(sorted(set(phashes))), thumb


def initial_group(name):
    stem = Path(name).stem
    if name.startswith("archive7__"):
        # Snapshot sering berurutan. Seluruh tanggal tetap satu kelompok,
        # kemudian kelompok tanggal dapat digabung oleh pasangan gambar mirip.
        match = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", stem)
        return "archive7:date:" + "-".join(match.groups()) if match else "archive7:undated_snapshots"
    return "archive6:original:" + re.sub(r"\.rf\.[0-9a-f]+$", "", stem, flags=re.I)


class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))

    def find(self, index):
        while self.parent[index] != index:
            self.parent[index] = self.parent[self.parent[index]]
            index = self.parent[index]
        return index

    def union(self, left, right):
        a, b = self.find(left), self.find(right)
        self.parent[max(a, b)] = min(a, b)


def hash_bands(value):
    # Jarak <=4 pada 64 bit menjamin minimal satu dari lima band identik.
    return [(band, (value >> shift) & ((1 << min(13, 64-shift)) - 1))
            for band, shift in enumerate(range(0, 64, 13))]


def group_images(records, hashes, thumbs, progress=print):
    union = UnionFind(len(records))
    named, exact, buckets = {}, {}, defaultdict(set)
    pairs = []
    for index, row in enumerate(records):
        for key, table in ((row["origin_group"], named), (row["pixel_sha256"], exact)):
            if key in table:
                union.union(index, table[key])
            else:
                table[key] = index
        candidates = set()
        for phash in hashes[index]:
            for band in hash_bands(phash):
                candidates.update(buckets[band])
        for other in sorted(candidates):
            distance = min((a ^ b).bit_count() for a in hashes[index] for b in hashes[other])
            if distance > 4:
                continue
            a = thumbs[index] - thumbs[index].mean()
            b = thumbs[other] - thumbs[other].mean()
            mae = min(float(np.abs(a - np.rot90(base, turns)).mean())
                      for base in (b, np.fliplr(b)) for turns in range(4))
            if mae <= 8:
                union.union(index, other)
                pairs.append(dict(image_a=records[other]["name"], image_b=row["name"],
                                  phash_distance=distance, centered_mae=round(mae, 5)))
        for phash in hashes[index]:
            for band in hash_bands(phash):
                buckets[band].add(index)
        if progress and (index+1) % 500 == 0:
            progress(f"Grouping: {index+1}/{len(records)} foto", flush=True)
    components = defaultdict(list)
    for index in range(len(records)):
        components[union.find(index)].append(index)
    groups = {}
    for indices in components.values():
        group_id = hashlib.sha256("\n".join(sorted(records[i]["name"] for i in indices)).encode()).hexdigest()[:20]
        groups[group_id] = indices
        for index in indices:
            records[index]["group_id"] = group_id
    return groups, pairs


def allocate_groups(records, groups, seed=42, ratios=(0.8, 0.1, 0.1)):
    if len(ratios) != 3 or min(ratios) <= 0 or not math.isclose(sum(ratios), 1, abs_tol=1e-8):
        raise ValueError("Isi tiga rasio positif dengan jumlah 1")
    if len(groups) < 3:
        raise ValueError("Kelompok independen terlalu sedikit untuk tiga split")
    source_names = sorted({row["source"] for row in records})
    source_totals = Counter(row["source"] for row in records)
    # Dimensi pertama jumlah seluruh foto; berikutnya jumlah per sumber.
    weights = [1.0] + [0.35] * len(source_names)
    totals = [len(records)] + [source_totals[s] for s in source_names]
    targets = [[ratio * total for total in totals] for ratio in ratios]
    sizes = {}
    for key, indices in groups.items():
        sources = Counter(records[i]["source"] for i in indices)
        sizes[key] = [len(indices)] + [sources[s] for s in source_names]
    rng = random.Random(seed)
    keys = sorted(groups)
    rng.shuffle(keys)
    keys.sort(key=lambda key: -sizes[key][0])
    counts = [[0] * len(totals) for _ in ratios]

    def score(split, values):
        return sum(weight * (value-target)**2 / max(1, target)
                   for weight, value, target in zip(weights, values, targets[split]))

    assignments = {}
    for key in keys:
        size = sizes[key]
        choices = list(range(3))
        rng.shuffle(choices)
        chosen = min(choices, key=lambda s: score(s, [a+b for a, b in zip(counts[s], size)]) - score(s, counts[s]))
        assignments[key] = chosen
        counts[chosen] = [a+b for a, b in zip(counts[chosen], size)]
    # Perbaiki rasio dengan memindahkan kelompok utuh; sumber ikut diseimbangkan.
    for _ in range(12):
        improved = False
        rng.shuffle(keys)
        for key in keys:
            old = assignments[key]
            size = sizes[key]
            if counts[old][0] == size[0]:
                continue
            alternatives = []
            for new in range(3):
                if new == old:
                    continue
                less = [a-b for a, b in zip(counts[old], size)]
                more = [a+b for a, b in zip(counts[new], size)]
                delta = score(old, less)+score(new, more)-score(old, counts[old])-score(new, counts[new])
                alternatives.append((delta, new, less, more))
            delta, new, less, more = min(alternatives)
            if delta < -1e-9:
                assignments[key] = new
                counts[old], counts[new] = less, more
                improved = True
        if not improved:
            break
    if min(row[0] for row in counts) <= 0:
        raise ValueError("Salah satu split kosong; butuh lebih banyak kelompok independen")
    for key, indices in groups.items():
        for index in indices:
            records[index]["split"] = SPLIT_NAMES[assignments[key]]
    return assignments


def scan_zip(source, progress=print):
    records, hashes, thumbs = [], [], []
    with zipfile.ZipFile(source) as archive:
        names = [i.filename for i in archive.infolist() if not i.is_dir()]
        if len(names) != len(set(names)):
            raise ValueError("Anggota ZIP duplikat")
        for name in names:
            p = PurePosixPath(name)
            if p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name:
                raise ValueError("Path ZIP tidak aman")
        yaml_text = archive.read("data.yaml").decode("utf-8-sig")
        if not re.search(r"names:\s*(?:\['cracks'\]|\[cracks\])", yaml_text):
            raise ValueError("Pakai ZIP crack v2 dengan satu kelas cracks")
        images = sorted(name for name in names if re.fullmatch(r"(?:train|valid|val|test)/images/[^/]+\.(?:jpg|jpeg|png)", name, re.I))
        if not images:
            raise ValueError("ZIP tidak memuat gambar train/valid/test")
        seen_names, seen_pixels = set(), {}
        for index, image_path in enumerate(images):
            p = PurePosixPath(image_path)
            name = p.name
            if name in seen_names:
                raise ValueError("Nama file sama di beberapa split")
            seen_names.add(name)
            split = p.parts[0]
            label_path = f"{split}/labels/{p.with_suffix('.txt').name}"
            raw_label = archive.read(label_path)
            boxes = validate_boxes(raw_label)
            raw = archive.read(image_path)
            width, height, pixel_hash, phashes, thumb = image_features(raw)
            if pixel_hash in seen_pixels:
                # Sumber v2 sudah dideduplikasi. Jangan memilih label konflik diam-diam.
                raise ValueError("Duplikat piksel masih ada: " + name + " / " + seen_pixels[pixel_hash])
            seen_pixels[pixel_hash] = name
            original = f"{split}/labels_original/{p.with_suffix('.txt').name}"
            records.append(dict(name=name, source_image=image_path, source_label=label_path,
                                source_original_label=original if original in names else "", previous_split=split,
                                source="archive7" if name.startswith("archive7__") else "archive6",
                                origin_group=initial_group(name), width=width, height=height,
                                boxes=len(boxes), pixel_sha256=pixel_hash,
                                image_sha256=hashlib.sha256(raw).hexdigest(), label_sha256=hashlib.sha256(raw_label).hexdigest()))
            hashes.append(phashes)
            thumbs.append(thumb)
            if progress and (index+1) % 500 == 0:
                progress(f"Audit: {index+1}/{len(images)} foto", flush=True)
    return records, hashes, thumbs


def dataset_fingerprint(records):
    lines = ["|".join(str(row[k]) for k in ("name", "split", "group_id", "image_sha256", "label_sha256"))
             for row in sorted(records, key=lambda r: r["name"])]
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()


def prepare_dataset(source_zip, destination, seed=42, ratios=(0.8, 0.1, 0.1), progress=print, allowed_sources=None, dataset_info=None):
    source_zip, destination = Path(source_zip), Path(destination)
    if destination.exists():
        raise FileExistsError("Folder output sudah ada; gunakan folder baru atau cache yang sudah diverifikasi")
    source_hash = file_sha256(source_zip)
    dataset_info = dict(dataset_info or {})
    records, hashes, thumbs = scan_zip(source_zip, progress)
    sources = sorted({row["source"] for row in records})
    if allowed_sources is not None and set(sources) != set(allowed_sources):
        raise ValueError(f"Sumber dataset {sources} tidak sesuai batasan {sorted(allowed_sources)}")
    groups, pairs = group_images(records, hashes, thumbs, progress)
    allocate_groups(records, groups, seed, ratios)
    fingerprint = dataset_fingerprint(records)
    for split in SPLIT_NAMES:
        for kind in ("images", "labels", "labels_original"):
            (destination / split / kind).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_zip) as archive:
        for row in records:
            new_image = f"{row['split']}/images/{row['name']}"
            label_name = PurePosixPath(row["name"]).with_suffix(".txt").name
            new_label = f"{row['split']}/labels/{label_name}"
            (destination / new_image).write_bytes(archive.read(row["source_image"]))
            (destination / new_label).write_bytes(archive.read(row["source_label"]))
            if row["source_original_label"]:
                (destination / row["split"] / "labels_original" / label_name).write_bytes(archive.read(row["source_original_label"]))
            row.update(image=new_image, label=new_label)
        # Salin atribusi serta laporan kurasi v2 tanpa menjalankan kode sumber.
        for member in archive.namelist():
            if member.startswith(("source_metadata/", "reports/")) and not member.endswith("/"):
                target = destination / "input_provenance" / member
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    (destination / "data.yaml").write_text("train: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames: ['cracks']\n", encoding="utf-8")
    fields = ["name", "source", "source_image", "source_label", "source_original_label", "previous_split",
              "origin_group", "group_id", "split", "image", "label", "width", "height", "boxes",
              "pixel_sha256", "image_sha256", "label_sha256"]
    save_csv(destination / "split_manifest.csv", records, fields)
    group_rows = [dict(group_id=key, images=len(indices), split=records[indices[0]]["split"],
                       sources=";".join(sorted({records[i]["source"] for i in indices})),
                       origin_groups=";".join(sorted({records[i]["origin_group"] for i in indices})))
                  for key, indices in sorted(groups.items())]
    save_csv(destination / "group_manifest.csv", group_rows, ["group_id", "images", "split", "sources", "origin_groups"])
    save_csv(destination / "similarity_pairs.csv", pairs, ["image_a", "image_b", "phash_distance", "centered_mae"])
    summary = []
    for split, target in zip(SPLIT_NAMES, ratios):
        selected = [r for r in records if r["split"] == split]
        summary.append(dict(split=split, images=len(selected), ratio=round(len(selected)/len(records), 6),
                            target_ratio=target, boxes=sum(r["boxes"] for r in selected),
                            groups=len({r["group_id"] for r in selected}),
                            archive6=sum(r["source"] == "archive6" for r in selected),
                            archive7=sum(r["source"] == "archive7" for r in selected)))
    save_csv(destination / "split_summary.csv", summary, list(summary[0]))
    config = dict(seed=seed, target_ratios=dict(zip(SPLIT_NAMES, ratios)), source_zip=source_zip.name,
                  source_sha256=source_hash, fingerprint=fingerprint, total_images=len(records),
                  dataset_sources=sources, source_image_counts=dict(Counter(row["source"] for row in records)),
                  total_groups=len(groups), similarity_pairs=len(pairs),
                  group_method="Roboflow original basename; archive7 whole capture-date/undated snapshots; union all near-duplicate pairs",
                  similarity_method="8 rotation/flip pHashes, Hamming <=4; mean-centered 64px MAE <=8",
                  split_method="Seeded weighted group allocation balanced by image count and source; no group divided",
                  limitation="Capture dates are inferred from filenames, not verified recording/location IDs. Similarity detection is heuristic. Existing archive6 augmentations remain together, including when a whole group is held out.",
                  summary=summary)
    if dataset_info:
        config["dataset_id"] = dataset_info["dataset_id"]
        config["original_source_sha256"] = dataset_info.get("original_source_sha256", "")
        save_json(destination / "dataset_info.json", dataset_info)
    save_json(destination / "split_config.json", config)
    archive6_only = sources == ["archive6"]
    photo_count = f"{len(records):,}".replace(",", ".")
    source_description = (f"Hanya kandidat retakan pada aspal jalan dari **archive 6** yang sudah lolos kurasi ({photo_count} foto). "
                          "Ini subset hasil kurasi, bukan seluruh foto mentah archive 6."
                          if archive6_only else "Gabungan crack archive 6 dan archive 7 yang sudah dikurasi (v2).")
    source_description = dataset_info.get("description", source_description)
    sequence_note = ("" if archive6_only else " Archive 7 dikelompokkan konservatif per tanggal dari nama file, dan snapshot tanpa tanggal tetap satu kelompok.")
    archive7_column = "" if archive6_only else " | Archive 7"
    archive7_divider = "" if archive6_only else " | ---:"
    table = "\n".join(f"| {r['split']} | {r['images']} | {r['ratio']:.1%} | {r['archive6']}" +
                      ("" if archive6_only else f" | {r['archive7']}") + " |" for r in summary)
    readme = f"""# Crack YOLO: split kelompok 80/10/10

Sumber: {source_description} Seluruh {len(records)} foto dibagi ulang bersama; split sebelumnya tidak dipertahankan. Satu kelas `0 = cracks`.

| Split | Gambar | Rasio aktual | Archive 6{archive7_column} |
| --- | ---: | ---: | ---:{archive7_divider} |
{table}

Seed: {seed}. Target train/valid/test: {ratios}. Rasio aktual mengikuti ukuran kelompok utuh. Skema A/B memakai split yang sama. Mengulangi seed dan sumber identik menghasilkan pembagian identik; urutan batch training tetap dapat diacak.

Kelompok: nama foto asal Roboflow (seluruh augmentasi tetap bersama).{sequence_note} Kelompok juga digabung jika ditemukan pasangan mirip dengan pHash delapan rotasi/flip, Hamming <=4 dan mean-centered thumbnail MAE <=8. Semua {len(pairs)} pasangan terdeteksi berada dalam split yang sama. Ini mengurangi risiko kebocoran; pemeriksaan kemiripan tidak menjamin independensi seluruh adegan/lokasi/rekaman.

Archive 6 sudah memiliki augmentasi offline; versi augmentasi dari satu foto tetap satu kelompok, termasuk bila masuk validasi/test. Metrik per gambar tidak menyatakan jumlah adegan independen; jumlah kelompok tersedia dalam manifest. Penilaian material aspal dan semantik anotasi tetap mengikuti batasan kurasi sumber. Dataset ini berisi foto positif; background tanpa retak belum diverifikasi.

Foto, bbox training, dan label asli disalin tanpa perubahan byte. `labels_original` masih memakai format/kelas sumber dan bukan label training. `input_provenance` memuat metadata/atribusi sumber serta laporan kurasi sebelumnya (laporan tersebut menjelaskan split lama, bukan split baru).

Gunakan `data.yaml` pada root untuk YOLO detect. Notebook baru memverifikasi hash dan manifest sebelum training, kemudian memakai split ini langsung. Tidak menjalankan training atau mengubah bobot model saat menyiapkan dataset ini.
"""
    (destination / "README.md").write_text(readme, encoding="utf-8")
    validation = verify_dataset(destination, expected_source_hash=source_hash)
    if progress:
        progress(json.dumps(dict(summary=summary, validation=validation)), flush=True)
    return config


def verify_dataset(destination, expected_source_hash=None):
    destination = Path(destination)
    config = json.loads((destination / "split_config.json").read_text(encoding="utf-8"))
    if expected_source_hash and config["source_sha256"] != expected_source_hash:
        raise ValueError("Cache memakai ZIP sumber berbeda")
    with (destination / "split_manifest.csv").open(encoding="utf-8", newline="") as stream:
        records = list(csv.DictReader(stream))
    if len(records) != config["total_images"] or dataset_fingerprint(records) != config["fingerprint"]:
        raise ValueError("Fingerprint manifest berubah")
    groups, pixels, by_name = defaultdict(set), defaultdict(set), {}
    for row in records:
        image = destination / row["image"]
        label = destination / row["label"]
        for relative in (row["image"], row["label"]):
            path = PurePosixPath(relative)
            if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
                raise ValueError("Path manifest tidak aman")
        if row["split"] not in SPLIT_NAMES or row["image"] != f"{row['split']}/images/{row['name']}" or row["label"] != f"{row['split']}/labels/{Path(row['name']).with_suffix('.txt').name}":
            raise ValueError("Path gambar/label tidak sesuai split")
        if file_sha256(image) != row["image_sha256"] or file_sha256(label) != row["label_sha256"]:
            raise ValueError("Isi gambar/label berubah: " + row["name"])
        if len(validate_boxes(label.read_bytes())) != int(row["boxes"]):
            raise ValueError("Jumlah bbox berubah")
        groups[row["group_id"]].add(row["split"])
        pixels[row["pixel_sha256"]].add(row["split"])
        if row["name"] in by_name:
            raise ValueError("Nama gambar duplikat")
        by_name[row["name"]] = row
    if any(len(splits) > 1 for splits in [*groups.values(), *pixels.values()]):
        raise ValueError("Kelompok/hash piksel bocor antar split")
    for split in SPLIT_NAMES:
        selected = [r for r in records if r["split"] == split]
        for kind, field in (("images", "image"), ("labels", "label")):
            actual = {p.relative_to(destination).as_posix() for p in (destination / split / kind).iterdir() if p.is_file()}
            if actual != {r[field] for r in selected}:
                raise ValueError("Daftar file tidak sesuai manifest")
    pair_count = 0
    with (destination / "similarity_pairs.csv").open(encoding="utf-8", newline="") as stream:
        for pair in csv.DictReader(stream):
            if by_name[pair["image_a"]]["split"] != by_name[pair["image_b"]]["split"]:
                raise ValueError("Pasangan mirip melintasi split")
            pair_count += 1
    if pair_count != config["similarity_pairs"]:
        raise ValueError("Jumlah pasangan kemiripan berbeda")
    report = dict(passed=True, images=len(records), groups=len(groups), known_cross_split_groups=0,
                  known_cross_split_pixel_hashes=0, known_cross_split_similar_pairs=0,
                  images_and_labels_hash_verified=True, all_labels_class_zero_bbox=True,
                  caveat="Pemeriksaan metadata dan kemiripan bersifat heuristik; bukan jaminan independensi lokasi/rekaman.")
    save_json(destination / "validation.json", report)
    return report


def package_dataset(destination, zip_path):
    destination, zip_path = Path(destination), Path(zip_path)
    if zip_path.exists():
        raise FileExistsError("Paket output sudah ada")
    with zipfile.ZipFile(zip_path, "x", compression=zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
        for path in sorted(destination.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(destination).as_posix())
    with zipfile.ZipFile(zip_path) as archive:
        bad = archive.testzip()
        if bad:
            raise ValueError("CRC ZIP gagal: " + bad)
    return dict(zip=zip_path.name, bytes=zip_path.stat().st_size, sha256=file_sha256(zip_path), crc_passed=True)


In [ ]:
with zipfile.ZipFile(DATA_ZIP) as archive:
    if 'dataset_info.json' not in archive.namelist():
        raise ValueError('Pakai roads_bridges_cracks_yolo26.zip yang label bbox-nya sudah diseragamkan.')
    SOURCE_DATASET_INFO = json.loads(archive.read('dataset_info.json'))
    if SOURCE_DATASET_INFO.get('dataset_id') != EXPECTED_DATASET_ID:
        raise ValueError('Notebook ini untuk seluruh Roads and Bridges Cracks, bukan subset/gabungan lain.')
# 4 · Baca split tersimpan, atau acak ulang kelompok dengan seed yang dipilih.
SOURCE_ZIP_SHA256 = file_sha256(DATA_ZIP)
if RESPLIT:
    signature = hashlib.sha256(json.dumps(dict(source=SOURCE_ZIP_SHA256, seed=SPLIT_SEED,
                                                ratios=SPLIT_RATIOS), sort_keys=True).encode()).hexdigest()[:16]
    YOLO_DATA_DIR = WORK_DIR / f'resplit_{signature}'
    if not YOLO_DATA_DIR.exists():
        with tempfile.TemporaryDirectory(prefix='prepare_', dir=WORK_DIR) as tmp:
            staged = Path(tmp) / 'dataset'
            prepare_dataset(DATA_ZIP, staged, seed=SPLIT_SEED, ratios=SPLIT_RATIOS, allowed_sources=EXPECTED_SOURCES, dataset_info=SOURCE_DATASET_INFO)
            staged.replace(YOLO_DATA_DIR)
    validation = verify_dataset(YOLO_DATA_DIR, expected_source_hash=SOURCE_ZIP_SHA256)
else:
    YOLO_DATA_DIR = WORK_DIR / f'stored_{SOURCE_ZIP_SHA256[:16]}'
    if not YOLO_DATA_DIR.exists():
        with tempfile.TemporaryDirectory(prefix='extract_', dir=WORK_DIR) as tmp:
            staged = Path(tmp) / 'dataset'
            staged.mkdir()
            with zipfile.ZipFile(DATA_ZIP) as archive:
                names = [x.filename for x in archive.infolist()]
                if len(names) != len(set(names)):
                    raise ValueError('Anggota ZIP duplikat')
                for name in names:
                    path = PurePosixPath(name)
                    if path.is_absolute() or '..' in path.parts or chr(92) in name or ':' in name:
                        raise ValueError('Path ZIP tidak aman')
                required = {'data.yaml', 'split_manifest.csv', 'split_config.json', 'similarity_pairs.csv'}
                if not required <= set(names):
                    raise ValueError('ZIP belum memiliki manifest split kelompok. Gunakan ZIP baru, atau RESPLIT=True untuk ZIP v2.')
                archive.extractall(staged)
            verify_dataset(staged)
            staged.replace(YOLO_DATA_DIR)
    validation = verify_dataset(YOLO_DATA_DIR)

SPLIT_CONFIG = json.loads((YOLO_DATA_DIR / 'split_config.json').read_text(encoding='utf-8'))
if SPLIT_CONFIG.get('dataset_id') != EXPECTED_DATASET_ID:
    raise ValueError('Identitas dataset dalam cache tidak sesuai Roads and Bridges Cracks lengkap.')
stored_ratios = tuple(SPLIT_CONFIG['target_ratios'][key] for key in SPLIT_NAMES)
if SPLIT_CONFIG['seed'] != SPLIT_SEED or stored_ratios != tuple(SPLIT_RATIOS):
    raise ValueError('Seed/rasio tidak cocok dengan split tersimpan. Aktifkan RESPLIT=True untuk membaginya ulang.')
config = yaml.safe_load((YOLO_DATA_DIR / 'data.yaml').read_text(encoding='utf-8'))
if config.get('names') != CLASS_NAMES or config.get('nc') != 1:
    raise ValueError('Dataset harus satu kelas cracks')

df = pd.read_csv(YOLO_DATA_DIR / 'split_manifest.csv', keep_default_na=False)
if set(df['source']) != EXPECTED_SOURCES or df['name'].str.startswith('archive7__').any():
    raise ValueError('Notebook ini hanya menerima Roads and Bridges Cracks. ZIP masih memuat sumber lain.')
if len(df) != EXPECTED_IMAGES:
    raise ValueError(f'Diharapkan {EXPECTED_IMAGES} foto positif unik Roads and Bridges Cracks, ditemukan {len(df)}.')
df['source_image_manifest'] = df['image']
df['source_label_manifest'] = df['label']
df['image'] = df['image'].map(lambda value: str((YOLO_DATA_DIR / value).resolve()))
df['label'] = df['label'].map(lambda value: str((YOLO_DATA_DIR / value).resolve()))
DATASET_FINGERPRINT = SPLIT_CONFIG['fingerprint']
DATASET = 'roads_bridges_cracks_full'
SPLIT = 'roads_bridges_group_random_80_10_10'
X_train, X_val, X_test = [df.loc[df['split'].eq(key), 'image'].tolist() for key in SPLIT_NAMES]

# YAML runtime memakai path absolut lokal; file YAML dalam ZIP tetap portable.
runtime_config = dict(path=str(YOLO_DATA_DIR.resolve()), train='train/images', val='valid/images',
                      test='test/images', nc=1, names=CLASS_NAMES)
runtime_yaml = WORK_DIR / f'runtime_{DATASET_FINGERPRINT[:16]}.yaml'
runtime_yaml.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding='utf-8')
DATA = str(runtime_yaml)
split_table = pd.DataFrame(SPLIT_CONFIG['summary'])
class_distribution = pd.DataFrame({key: [int(df.loc[df['split'].eq(key), 'boxes'].sum()),
                                        int(df['split'].eq(key).sum())] for key in SPLIT_NAMES},
                                  index=['cracks_bbox', 'Jumlah gambar'])
print('Validasi dataset:', validation)
display(split_table.drop(columns=['archive7'], errors='ignore'))
display(class_distribution)
print('Train / valid / test:', len(X_train), len(X_val), len(X_test))
print('Fingerprint:', DATASET_FINGERPRINT)

# Simpan identitas split ke Drive untuk melanjutkan eksperimen dari runtime baru.
SPLIT_REPORT_DIR = DRIVE_PROJECT_DIR / 'data' / 'split_reports' / DATASET_FINGERPRINT
SPLIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)
for name in ['split_config.json', 'split_manifest.csv', 'group_manifest.csv',
             'split_summary.csv', 'similarity_pairs.csv', 'validation.json', 'README.md']:
    shutil.copy2(YOLO_DATA_DIR / name, SPLIT_REPORT_DIR / name)
print('Laporan split tersimpan di', SPLIT_REPORT_DIR)


In [ ]:
# 5 · Lihat contoh bbox dari setiap split sebelum training.
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for row_index, split in enumerate(SPLIT_NAMES):
    sample = df.loc[df['split'].eq(split)].sample(n=min(3, int(df['split'].eq(split).sum())), random_state=SPLIT_SEED)
    for ax in axes[row_index]:
        ax.axis('off')
    for ax, record in zip(axes[row_index], sample.itertuples(index=False)):
        with Image.open(record.image) as opened:
            image = opened.convert('RGB')
        ax.imshow(image)
        for _, x, y, w, h in validate_boxes(Path(record.label).read_bytes()):
            ax.add_patch(Rectangle(((x-w/2)*image.width, (y-h/2)*image.height), w*image.width,
                                   h*image.height, fill=False, edgecolor='cyan', linewidth=1.4))
        ax.set_title(f'{split} | {record.source} | {record.boxes} bbox')
plt.tight_layout()
plt.show()


## 6. Training YOLO26s

**Skema A:** augmentasi bawaan YOLO. **Skema B:** ditambah perubahan kecerahan/kontras, silau, dan bayangan. Augmentasi online hanya untuk train. Validasi memilih checkpoint `best.pt`; metrik test dilaporkan terpisah. Pemilihan skema untuk inferensi memakai skor **validasi**, bukan skor test.

Split/fingerprint yang sama dipakai oleh kedua skema. Run yang selesai dilewati; run terputus dilanjutkan dari `last.pt` setelah konfigurasi diperiksa. Ganti `RUN_SUFFIX` untuk eksperimen baru. Resume memerlukan dataset serta path runtime yang sama; mode sumber, nama ZIP, dan konfigurasi split sebaiknya dipertahankan saat pindah runtime.


In [ ]:
# 6 · Lokasi run serta identitas konfigurasi.
RUN_NAMES = {skema: f'{Path(MODEL).stem}_{skema}_roads_bridges_cracks_s{SPLIT_SEED}_{DATASET_FINGERPRINT[:8]}_{RUN_SUFFIX}'
             for skema in SKEMA_LIST}

def run_paths(skema):
    folder = RUNS_ROOT / RUN_NAMES[skema]
    return folder, folder / 'weights' / 'best.pt', folder / 'weights' / 'last.pt'

def run_done(skema):
    folder, best, last = run_paths(skema)
    return all(path.is_file() for path in [best, last, folder / 'metrics.csv', folder / 'predictions_test.csv'])

def run_config(skema):
    return dict(model=MODEL, classes=CLASS_NAMES, skema=skema, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                epochs=EPOCHS, patience=PATIENCE, close_mosaic=CLOSE_MOSAIC, optimizer='auto',
                train_seed=TRAIN_SEED, split_seed=SPLIT_SEED, ratios=list(SPLIT_RATIOS),
                split=SPLIT, dataset_fingerprint=DATASET_FINGERPRINT,
                ultralytics=ultralytics.__version__)

def prepare_run(skema):
    folder, best, last = run_paths(skema)
    path = folder / 'config.json'
    config = run_config(skema)
    if folder.exists():
        if not path.is_file() or json.loads(path.read_text(encoding='utf-8')) != config:
            raise RuntimeError(f'{folder} memakai konfigurasi lain. Ganti RUN_SUFFIX untuk run baru.')
        return
    folder.mkdir(parents=True, exist_ok=False)
    path.write_text(json.dumps(config, indent=2), encoding='utf-8')
    for name in ['split_config.json', 'split_manifest.csv', 'group_manifest.csv',
                 'split_summary.csv', 'similarity_pairs.csv', 'validation.json']:
        shutil.copy2(YOLO_DATA_DIR / name, folder / name)

for skema in SKEMA_LIST:
    folder, best, last = run_paths(skema)
    print(skema, RUN_NAMES[skema], ':', 'selesai' if run_done(skema) else ('akan resume' if last.is_file() else 'baru'))


In [ ]:
# 7 · Augmentasi, training, riwayat, dan evaluasi.
def build_augmentations(skema):
    if skema == 'A':
        return None
    import albumentations as A
    return [
        A.Blur(p=0.01), A.MedianBlur(p=0.01), A.ToGray(p=0.01), A.CLAHE(p=0.01),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.RandomSunFlare(flare_roi=(0, 0, 1, 0.5)),
            A.RandomShadow(shadow_roi=(0, 0.5, 1, 1)),
        ], p=0.5),
    ]

def train_run(skema):
    folder, best, last = run_paths(skema)
    if last.is_file():
        model = YOLO(str(last))
        if model.ckpt.get('epoch', -1) >= 0:
            print('Melanjutkan training dari', last)
            model.train(resume=True, data=DATA, device=DEVICE)
        else:
            print('Checkpoint selesai; melanjutkan ke evaluasi.')
    else:
        model = YOLO(MODEL)
        model.info(detailed=False, verbose=True)
        options = dict(data=DATA, epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMG_SIZE,
                       device=DEVICE, workers=WORKERS, seed=TRAIN_SEED, deterministic=True,
                       optimizer='auto', patience=PATIENCE, close_mosaic=CLOSE_MOSAIC,
                       save=True, val=True, plots=True, project=str(RUNS_ROOT),
                       name=RUN_NAMES[skema], exist_ok=True)
        augmentations = build_augmentations(skema)
        if augmentations is not None:
            options['augmentations'] = augmentations
        print('Training skema', skema, '| split seed', SPLIT_SEED, '| train seed', TRAIN_SEED)
        model.train(**options)
    del model
    if not best.is_file() or not last.is_file():
        raise FileNotFoundError('Checkpoint training belum lengkap')
    print('Best model:', best)
    print('Last checkpoint:', last)

def show_training(skema):
    folder, best, last = run_paths(skema)
    history = pd.read_csv(folder / 'results.csv')
    history.columns = history.columns.str.strip()
    display(history.tail())
    graph = folder / 'results.png'
    if graph.is_file():
        with Image.open(graph) as image:
            display(image.copy())

def evaluate_run(skema):
    folder, best, last = run_paths(skema)
    model = YOLO(str(best))
    results = {}
    for split in ['val', 'test']:
        metric = model.val(data=DATA, split=split, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                           workers=WORKERS, device=DEVICE, conf=0.001, iou=0.7,
                           plots=True, project=str(folder), name=f'eval_{split}', exist_ok=True)
        results[split] = dict(precision=float(metric.box.mp), recall=float(metric.box.mr),
                              mAP50=float(metric.box.map50), mAP50_95=float(metric.box.map))
    row = dict(model=Path(MODEL).stem, skema=skema, checkpoint='best.pt',
               jumlah_training=len(X_train), jumlah_validation=len(X_val), jumlah_testing=len(X_test),
               split_seed=SPLIT_SEED, dataset_fingerprint=DATASET_FINGERPRINT,
               **{f'{split}_{name}': value for split, values in results.items() for name, value in values.items()})
    metrics = pd.DataFrame([row])
    metrics.to_csv(folder / 'metrics.csv', index=False)
    display(metrics.round(5))

    predictions = []
    stream = model.predict(source=str(YOLO_DATA_DIR / 'test' / 'images'), imgsz=IMG_SIZE,
                           conf=PREDICT_CONF, iou=0.7, device=DEVICE, half=False, verbose=False, stream=True)
    for result in tqdm(stream, total=len(X_test), desc='Prediksi test', unit='foto'):
        if result.boxes is None or len(result.boxes) == 0:
            predictions.append(dict(image=result.path, detected=False, class_id=None, kelas=None,
                                    confidence=None, x1=None, y1=None, x2=None, y2=None))
            continue
        for box, score, cls in zip(result.boxes.xyxy.cpu().tolist(), result.boxes.conf.cpu().tolist(),
                                   result.boxes.cls.cpu().tolist()):
            predictions.append(dict(image=result.path, detected=True, class_id=int(cls), kelas=CLASS_NAMES[int(cls)],
                                    confidence=score, **dict(zip(['x1', 'y1', 'x2', 'y2'], box))))
    pd.DataFrame(predictions).to_csv(folder / 'predictions_test.csv', index=False)
    df.loc[df['split'].eq('test')].to_csv(folder / 'ground_truth_test_manifest.csv', index=False)
    del model
    print('Metrik, prediksi bbox, dan ground truth test tersimpan di', folder)


In [ ]:
# 8 · Jalankan skema. Sel ini benar-benar memulai training GPU.
for skema in SKEMA_LIST:
    print('Skema', skema, '|', RUN_NAMES[skema])
    prepare_run(skema)
    if run_done(skema):
        print('Run sudah selesai, dilewati.')
        continue
    train_run(skema)
    show_training(skema)
    evaluate_run(skema)
    gc.collect()
    torch.cuda.empty_cache()
print('Semua skema terpilih selesai.')


## 9. Ringkasan dan tes foto sendiri

mAP50 dan mAP50-95 adalah metrik deteksi bbox. Confidence inferensi default 0,25; evaluasi mAP memakai 0,001. Metrik test tidak dipakai untuk memilih skema. Semua gambar dataset ini positif crack; performa pada jalan tanpa crack perlu diuji menggunakan foto negatif terpisah yang dianotasi/diverifikasi.


In [ ]:
summary = pd.concat([pd.read_csv(run_paths(skema)[0] / 'metrics.csv') for skema in SKEMA_LIST], ignore_index=True)
summary.insert(0, 'run', [RUN_NAMES[skema] for skema in SKEMA_LIST])
summary_path = RUNS_ROOT / f'perbandingan_{Path(MODEL).stem}_roads_bridges_cracks_{DATASET_FINGERPRINT[:8]}_{RUN_SUFFIX}.csv'
summary.to_csv(summary_path, index=False)
display(summary[['run', 'skema', 'val_mAP50_95', 'test_precision', 'test_recall',
                 'test_mAP50', 'test_mAP50_95']].round(4))
INFERENCE_SKEMA = summary.loc[summary['val_mAP50_95'].idxmax(), 'skema']
print('Skema untuk inferensi berdasarkan VALIDASI:', INFERENCE_SKEMA)
print('Ringkasan:', summary_path)


In [ ]:
# 10 · Inference contoh gambar (opsional, setelah training).
inference_model = YOLO(str(run_paths(INFERENCE_SKEMA)[1]))

def predict_crack(image_path, confidence=PREDICT_CONF):
    result = inference_model.predict(source=str(image_path), imgsz=IMG_SIZE, conf=confidence,
                                     device=DEVICE, half=False, verbose=False)[0]
    print('Gambar:', image_path)
    if result.boxes is None or len(result.boxes) == 0:
        print('Tidak ada crack terdeteksi pada threshold ini.')
    else:
        for cls, score, box in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist(), result.boxes.xyxy.tolist()):
            print('Label:', result.names[int(cls)], '| Confidence:', f'{score:.2%}',
                  '| bbox:', [round(value, 1) for value in box])
    plt.figure(figsize=(12, 6))
    plt.imshow(result.plot()[..., ::-1])
    plt.axis('off')
    plt.show()
    return result

CONTOH_GAMBAR = X_test[0]  # Ganti dengan path foto sendiri jika diinginkan.
contoh_hasil = predict_crack(CONTOH_GAMBAR)


In [ ]:
# 11 · Upload foto sendiri untuk inference (opsional).
# Aktifkan RUN_UPLOAD_TEST jika ingin dialog upload muncul.
RUN_UPLOAD_TEST = False
if RUN_UPLOAD_TEST:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        predict_crack(filename)


Hasil tersedia di `MyDrive/deteksi-jalan/hasil/yolo_roads_bridges_cracks/<RUN_NAME>/`:

| File | Isi |
| --- | --- |
| `weights/best.pt`, `weights/last.pt` | Model terbaik berdasarkan validasi dan checkpoint resume |
| `results.csv`, `results.png` | Riwayat serta grafik training |
| `eval_val/`, `eval_test/` | Grafik evaluasi, confusion matrix, dan kurva PR |
| `metrics.csv` | Precision, recall, mAP50, dan mAP50-95 validasi/test |
| `predictions_test.csv` | Prediksi bbox pada test |
| `split_manifest.csv`, `group_manifest.csv` | Pembagian foto dan kelompok untuk audit |
| `split_config.json`, `validation.json`, `config.json` | Seed, rasio, fingerprint, dan konfigurasi run |

Sumber/atribusi dataset berada dalam ZIP di `input_provenance/source_metadata/`. Dataset pothole dan notebook lama tidak diubah. Jangan membandingkan metrik split baru langsung dengan metrik split lama sebagai perbandingan model yang setara.
